# Principio de Abierto/Cerrado (OCP) — Sistema de citas médicas

## Introducción
El Principio de Abierto/Cerrado (OCP) establece que las entidades de software deben estar **abiertas a extensión pero cerradas a modificación**: agregar un nuevo caso no debería requerir editar código existente y ya probado.

## Objetivos
- Mostrar un cálculo de costos que crece con `if/elif` cada vez que aparece un nuevo tipo de consulta.
- Rediseñarlo con polimorfismo para poder agregar un nuevo tipo de consulta **sin modificar** el código existente.

## Ejemplo que viola el OCP

`CalculadoraCosto` calcula el costo según el tipo de consulta usando una cadena de `if/elif`.

In [1]:
class CalculadoraCosto:
    def calcular(self, tipo_consulta: str, duracion_minutos: int) -> float:
        if tipo_consulta == "general":
            return 50000 + duracion_minutos * 500
        elif tipo_consulta == "especialista":
            return 120000 + duracion_minutos * 1000
        elif tipo_consulta == "urgencia":
            return 200000 + duracion_minutos * 1500
        else:
            raise ValueError(f"Tipo de consulta no soportado: {tipo_consulta}")

    def describir(self, tipo_consulta: str) -> str:
        return f"Cálculo de costo para consulta tipo '{tipo_consulta}'"


calculadora = CalculadoraCosto()
print(calculadora.describir("especialista"))
print(calculadora.calcular("especialista", 30))

Cálculo de costo para consulta tipo 'especialista'
150000


### Por qué esto es un problema

Si la clínica agrega un nuevo tipo de consulta (por ejemplo "telemedicina"), la **única** forma de soportarlo es abrir `CalculadoraCosto` y agregar un nuevo `elif`. Eso significa:

- Editar una clase que ya estaba funcionando y probada, arriesgando romper los casos existentes.
- La cadena `if/elif` crece indefinidamente con cada tipo nuevo.

`CalculadoraCosto` no está cerrada a modificación: cada caso nuevo obliga a tocarla.

## Versión corregida: extensión sin modificación

Se define una abstracción `TipoConsulta` con el método `calcular_costo`. Cada tipo de consulta es una subclase independiente. `ProcesadorFacturacion` (la clase de "alto nivel") solo conoce la abstracción, nunca los tipos concretos.

In [2]:
from abc import ABC, abstractmethod


class TipoConsulta(ABC):
    def __init__(self, nombre: str, tarifa_base: float) -> None:
        self.nombre = nombre
        self.tarifa_base = tarifa_base

    @abstractmethod
    def calcular_costo(self, duracion_minutos: int) -> float:
        ...

    def describir(self) -> str:
        return f"Cálculo de costo para consulta tipo '{self.nombre}'"


class ConsultaGeneral(TipoConsulta):
    def __init__(self) -> None:
        super().__init__(nombre="general", tarifa_base=50000)

    def calcular_costo(self, duracion_minutos: int) -> float:
        return self.tarifa_base + duracion_minutos * 500


class ConsultaEspecialista(TipoConsulta):
    def __init__(self) -> None:
        super().__init__(nombre="especialista", tarifa_base=120000)

    def calcular_costo(self, duracion_minutos: int) -> float:
        return self.tarifa_base + duracion_minutos * 1000


class ConsultaUrgencia(TipoConsulta):
    def __init__(self) -> None:
        super().__init__(nombre="urgencia", tarifa_base=200000)

    def calcular_costo(self, duracion_minutos: int) -> float:
        return self.tarifa_base + duracion_minutos * 1500


class ProcesadorFacturacion:
    def __init__(self) -> None:
        self.historial = []

    def procesar(self, tipo_consulta: TipoConsulta, duracion_minutos: int) -> float:
        costo = tipo_consulta.calcular_costo(duracion_minutos)
        self.historial.append((tipo_consulta.nombre, costo))
        return costo

    def total_procesado(self) -> float:
        return sum(costo for _, costo in self.historial)

In [3]:
procesador = ProcesadorFacturacion()
print(procesador.procesar(ConsultaEspecialista(), 30))
print(procesador.procesar(ConsultaGeneral(), 20))
print("Total procesado:", procesador.total_procesado())

150000
60000
Total procesado: 210000


### Demostración: agregar un caso nuevo sin tocar código existente

Ahora la clínica quiere ofrecer **telemedicina**. Se agrega una subclase nueva; **no se modifica ni una línea** de `TipoConsulta`, `ProcesadorFacturacion` ni de los tipos ya existentes.

In [4]:
class ConsultaTelemedicina(TipoConsulta):
    def __init__(self) -> None:
        super().__init__(nombre="telemedicina", tarifa_base=30000)

    def calcular_costo(self, duracion_minutos: int) -> float:
        return self.tarifa_base + duracion_minutos * 300


# ProcesadorFacturacion no cambió: sigue funcionando con el tipo nuevo sin editarlo.
costo_telemedicina = procesador.procesar(ConsultaTelemedicina(), 15)
print("Costo telemedicina:", costo_telemedicina)
print("Total procesado tras agregar el nuevo tipo:", procesador.total_procesado())

assert costo_telemedicina == 30000 + 15 * 300
print("¡Todo correcto!")

Costo telemedicina: 34500
Total procesado tras agregar el nuevo tipo: 244500
¡Todo correcto!


### Análisis

- `ProcesadorFacturacion` depende únicamente de la abstracción `TipoConsulta`, nunca de un tipo concreto.
- Agregar `ConsultaTelemedicina` no requirió editar `TipoConsulta`, `ProcesadorFacturacion` ni ninguna de las consultas existentes: el sistema está **abierto a extensión** (se agregó una clase nueva) y **cerrado a modificación** (nada existente se tocó).
- Si se hubiera usado el enfoque `if/elif`, este mismo caso habría exigido editar `CalculadoraCosto`.

## Autoevaluación
- ¿Qué pasaría si mañana piden un descuento del 10% solo para adultos mayores en consultas generales? ¿Dónde se implementaría sin tocar las demás clases?

## Referencias
- Martin, R. C. — *Design Principles and Design Patterns* (el paper original del OCP).
- [SOLID Principles en Python – Real Python](https://realpython.com/solid-principles-python/)